# M5 — Data Modelling & Visualisation: MovieLens 100K

Train and evaluate three rating-prediction models on `u1.base` / `u1.test`, compare with RMSE, MAE, Precision@10, and Recall@10, and export stakeholder charts.


## Setup — imports, model definitions, and plotting helpers


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
K_FACTORS = 40
K_NEIGHBORS = 30
TOP_N = 10
SVD_ITER = 15


def resolve_paths():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "M5"]
    for m5_dir in candidates:
        root = m5_dir.parent
        for raw_dir in [root / "M1" / "ml-100k", root / "M2 & M3" / "data" / "raw" / "ml-100k"]:
            if (raw_dir / "u1.base").exists() and (raw_dir / "u1.test").exists():
                fig_dir = m5_dir / "figures"
                fig_dir.mkdir(parents=True, exist_ok=True)
                return m5_dir, raw_dir
    raise FileNotFoundError("Cannot find u1.base / u1.test under M1 or M2 & M3.")


def load_split(raw_dir):
    cols = ["user_id", "item_id", "rating", "timestamp"]
    train = pd.read_csv(raw_dir / "u1.base", sep="\t", names=cols)
    test = pd.read_csv(raw_dir / "u1.test", sep="\t", names=cols)
    return train, test


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))


def clip_ratings(values):
    return np.clip(values, 1.0, 5.0)


class BiasBaseline:
    name = "Bias Baseline (μ + user + item)"

    def fit(self, train):
        self.mu = float(train["rating"].mean())
        self.user_bias = train.groupby("user_id")["rating"].mean() - self.mu
        self.item_bias = train.groupby("item_id")["rating"].mean() - self.mu

    def predict_row(self, user_id, item_id):
        bu = self.user_bias.get(user_id, 0.0)
        bi = self.item_bias.get(item_id, 0.0)
        return float(clip_ratings(np.array([self.mu + bu + bi]))[0])

    def predict_frame(self, frame):
        users = frame["user_id"].map(self.user_bias).fillna(0.0).to_numpy()
        items = frame["item_id"].map(self.item_bias).fillna(0.0).to_numpy()
        return clip_ratings(self.mu + users + items)


class ItemKNN:
    name = f"Item-Based CF (k={K_NEIGHBORS})"

    def __init__(self, k=K_NEIGHBORS):
        self.k = k

    def fit(self, train):
        self.train_lookup = {(r.user_id, r.item_id): r.rating for r in train.itertuples()}
        self.mu = float(train["rating"].mean())
        self.user_means = train.groupby("user_id")["rating"].mean()
        self.item_means = train.groupby("item_id")["rating"].mean()
        users = np.sort(train["user_id"].unique())
        items = np.sort(train["item_id"].unique())
        self.user_index = {u: i for i, u in enumerate(users)}
        self.item_index = {it: i for i, it in enumerate(items)}
        self.index_item = {i: it for it, i in self.item_index.items()}
        rows = train["user_id"].map(self.user_index).to_numpy()
        cols = train["item_id"].map(self.item_index).to_numpy()
        centered = train["rating"].to_numpy() - train["user_id"].map(self.user_means).to_numpy()
        matrix = csr_matrix((centered, (rows, cols)), shape=(len(users), len(items)))
        denom = np.sqrt(matrix.power(2).sum(axis=0)).A1
        denom[denom == 0] = 1.0
        sim = (matrix.T @ matrix).toarray()
        sim = sim / np.outer(denom, denom)
        np.fill_diagonal(sim, 0.0)
        self.sim = sim
        self.train_pairs = set(zip(train["user_id"], train["item_id"]))

    def predict_row(self, user_id, item_id):
        if item_id not in self.item_index or user_id not in self.user_index:
            return float(self.item_means.get(item_id, self.mu))
        i_idx = self.item_index[item_id]
        user_mean = self.user_means.get(user_id, self.mu)
        sims = self.sim[i_idx]
        rated_items = [j for j in range(len(sims)) if (user_id, self.index_item[j]) in self.train_pairs]
        if not rated_items:
            return float(self.item_means.get(item_id, self.mu))
        rated_items = np.array(rated_items, dtype=int)
        neighbor_sims = sims[rated_items]
        order = np.argsort(-np.abs(neighbor_sims))[: self.k]
        chosen = rated_items[order]
        weights = neighbor_sims[order]
        if np.allclose(weights, 0):
            return float(self.item_means.get(item_id, self.mu))
        neighbor_ratings = np.array([
            self.train_lookup.get((user_id, self.index_item[j]), self.mu) for j in chosen
        ], dtype=float)
        pred = user_mean + np.dot(weights, neighbor_ratings - user_mean) / (np.sum(np.abs(weights)) + 1e-8)
        return float(clip_ratings(np.array([pred]))[0])

    def predict_frame(self, frame):
        return np.array([self.predict_row(int(u), int(i)) for u, i in zip(frame["user_id"], frame["item_id"])])


class MatrixFactorizationSVD:
    name = f"Matrix Factorization (SVD, k={K_FACTORS})"

    def fit(self, train):
        self.mu = float(train["rating"].mean())
        self.user_bias = train.groupby("user_id")["rating"].mean() - self.mu
        self.item_bias = train.groupby("item_id")["rating"].mean() - self.mu
        users = np.sort(train["user_id"].unique())
        items = np.sort(train["item_id"].unique())
        self.user_index = {u: i for i, u in enumerate(users)}
        self.item_index = {it: i for i, it in enumerate(items)}
        rows = train["user_id"].map(self.user_index).to_numpy()
        cols = train["item_id"].map(self.item_index).to_numpy()
        residuals = (
            train["rating"].to_numpy()
            - self.mu
            - train["user_id"].map(self.user_bias).fillna(0.0).to_numpy()
            - train["item_id"].map(self.item_bias).fillna(0.0).to_numpy()
        )
        matrix = csr_matrix((residuals, (rows, cols)), shape=(len(users), len(items)))
        n_components = min(K_FACTORS, min(matrix.shape) - 1)
        svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE, n_iter=SVD_ITER)
        self.user_factors = svd.fit_transform(matrix)
        self.item_factors = svd.components_.T

    def predict_row(self, user_id, item_id):
        bu = self.user_bias.get(user_id, 0.0)
        bi = self.item_bias.get(item_id, 0.0)
        if user_id not in self.user_index or item_id not in self.item_index:
            return float(clip_ratings(np.array([self.mu + bu + bi]))[0])
        latent = float(self.user_factors[self.user_index[user_id]] @ self.item_factors[self.item_index[item_id]])
        return float(clip_ratings(np.array([self.mu + bu + bi + latent]))[0])

    def predict_frame(self, frame):
        return np.array([self.predict_row(int(u), int(i)) for u, i in zip(frame["user_id"], frame["item_id"])])


def precision_recall_at_k(model, train, test, k=TOP_N, relevance_threshold=4.0):
    train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()
    test_relevant = test[test["rating"] >= relevance_threshold].groupby("user_id")["item_id"].apply(set).to_dict()
    all_items = set(train["item_id"].unique())
    precisions, recalls = [], []
    for user_id, relevant in test_relevant.items():
        if not relevant:
            continue
        candidates = list(all_items - train_items_by_user.get(user_id, set()))
        if not candidates:
            continue
        scores = np.array([model.predict_row(int(user_id), int(it)) for it in candidates])
        recommended = {candidates[i] for i in np.argsort(-scores)[:k]}
        hits = len(recommended & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))
    return float(np.mean(precisions)), float(np.mean(recalls))


def evaluate_model(model, train, test):
    preds = model.predict_frame(test)
    y_true = test["rating"].to_numpy(dtype=float)
    p_at_k, r_at_k = precision_recall_at_k(model, train, test)
    return {
        "model": model.name,
        "rmse": rmse(y_true, preds),
        "mae": mae(y_true, preds),
        f"precision@{TOP_N}": p_at_k,
        f"recall@{TOP_N}": r_at_k,
        "predictions": preds,
    }


def plot_model_comparison(results, fig_dir):
    metrics = ["rmse", "mae", f"precision@{TOP_N}", f"recall@{TOP_N}"]
    plot_df = results.melt(id_vars="model", value_vars=metrics, var_name="metric", value_name="score")
    labels = {
        "rmse": "RMSE (lower is better)",
        "mae": "MAE (lower is better)",
        f"precision@{TOP_N}": f"Precision@{TOP_N} (higher is better)",
        f"recall@{TOP_N}": f"Recall@{TOP_N} (higher is better)",
    }
    plot_df["metric_label"] = plot_df["metric"].map(labels)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, subset in zip(
        axes,
        [
            plot_df[plot_df["metric"].isin(["rmse", "mae"])],
            plot_df[plot_df["metric"].isin([f"precision@{TOP_N}", f"recall@{TOP_N}"])],
        ],
    ):
        sns.barplot(data=subset, x="model", y="score", hue="metric_label", ax=ax)
        ax.set_xlabel("")
        ax.set_ylabel("Score")
        ax.tick_params(axis="x", rotation=15)
        ax.legend(title="", fontsize=8)
    fig.suptitle("Model Performance Comparison on MovieLens 100K (u1.test)", y=1.02)
    fig.tight_layout()
    fig.savefig(fig_dir / "01_model_comparison.png", bbox_inches="tight")
    plt.close(fig)


def plot_prediction_quality(test, preds, model_name, fig_dir):
    errors = preds - test["rating"].to_numpy(dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(test["rating"], preds, alpha=0.15, s=8, color="#2a6f97")
    axes[0].plot([1, 5], [1, 5], "--", color="#e76f51", linewidth=1.5)
    axes[0].set(xlabel="Actual Rating", ylabel="Predicted Rating", title=f"Predicted vs Actual — {model_name}", xlim=(0.8, 5.2), ylim=(0.8, 5.2))
    sns.histplot(errors, bins=30, kde=True, ax=axes[1], color="#264653")
    axes[1].axvline(0, color="#e76f51", linestyle="--")
    axes[1].set(xlabel="Prediction Error (predicted − actual)", title="Error Distribution")
    fig.tight_layout()
    fig.savefig(fig_dir / "02_prediction_quality_mf.png", bbox_inches="tight")
    plt.close(fig)


def plot_stakeholder_lift(results, fig_dir):
    baseline_rmse = results.loc[results["model"].str.contains("Bias"), "rmse"].iloc[0]
    lift = results.copy()
    lift["rmse_reduction_pct"] = (baseline_rmse - lift["rmse"]) / baseline_rmse * 100
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(lift["model"], lift["rmse_reduction_pct"], color=["#adb5bd", "#6c757d", "#2a9d8f"])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set(xlabel="RMSE Improvement vs Bias Baseline (%)", title="How Much Better Are Advanced Models?")
    for bar, val in zip(bars, lift["rmse_reduction_pct"]):
        ax.text(max(val + 0.3, 0.2), bar.get_y() + bar.get_height() / 2, f"{val:.1f}%", va="center", fontsize=10)
    fig.tight_layout()
    fig.savefig(fig_dir / "03_stakeholder_rmse_lift.png", bbox_inches="tight")
    plt.close(fig)


def plot_recommendation_preview(mf_model, train, test, raw_dir, fig_dir, sample_user=196):
    names = ["item_id", "title", "release", "video", "url"] + [f"g{i}" for i in range(19)]
    items = pd.read_csv(raw_dir / "u.item", sep="|", header=None, encoding="latin-1", names=names)
    candidates = list(set(train["item_id"].unique()) - set(train.loc[train["user_id"] == sample_user, "item_id"]))
    scores = np.array([mf_model.predict_row(sample_user, it) for it in candidates])
    top_idx = np.argsort(-scores)[:8]
    top_items = [candidates[i] for i in top_idx]
    top_scores = scores[top_idx]
    actual_test = test[test["user_id"] == sample_user].merge(items[["item_id", "title"]], on="item_id", how="left")
    fig, ax = plt.subplots(figsize=(10, 6))
    y_labels = [items.loc[items["item_id"] == it, "title"].iloc[0][:45] for it in top_items][::-1]
    ax.barh(y_labels, top_scores[::-1], color="#457b9d")
    ax.set(xlabel="Predicted Rating", xlim=(3.0, 5.0))
    ax.set_title(f"Top Recommended Movies for User {sample_user}\n(Model: Matrix Factorization — not yet seen in training)")
    note = "Held-out ratings for this user:\n" + "".join(f"• {str(r.title)[:40]}: actual {r.rating}\n" for r in actual_test.itertuples())
    ax.text(1.02, 0.5, note, transform=ax.transAxes, fontsize=8, va="center", bbox=dict(boxstyle="round", facecolor="#f1faee", alpha=0.9))
    fig.tight_layout()
    fig.savefig(fig_dir / "04_recommendation_preview.png", bbox_inches="tight")
    plt.close(fig)


M5_DIR, RAW_DIR = resolve_paths()
FIG_DIR = M5_DIR / "figures"
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150})
print(f"M5_DIR: {M5_DIR}\nRAW_DIR: {RAW_DIR}")


M5_DIR: C:\Users\admin\Desktop\数据科学项目\Work\M5
RAW_DIR: C:\Users\admin\Desktop\数据科学项目\Work\M1\ml-100k


## 1. Load train / test split


In [2]:
train, test = load_split(RAW_DIR)
print(f"Train: {len(train):,} | Test: {len(test):,}")
train.head()


Train: 80,000 | Test: 20,000


,user_id,item_id,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,3,4,878542960
3,1,4,3,876893119
4,1,5,3,889751712


## 2. Model 1 — Bias Baseline

**Rationale:** Interpretable lower bound capturing user leniency and item quality.


In [3]:
baseline = BiasBaseline()
baseline.fit(train)
base_metrics = evaluate_model(baseline, train, test)
base_preds = base_metrics.pop('predictions')
pd.Series({k: round(v, 4) for k, v in base_metrics.items() if k != 'model'})


rmse            0.9772
mae             0.7662
precision@10    0.0090
recall@10       0.0031
dtype: float64

## 3. Model 2 — Item-Based CF (k=30)

**Rationale:** Classic neighbourhood baseline from the project proposal.


In [4]:
item_cf = ItemKNN()
item_cf.fit(train)
cf_metrics = evaluate_model(item_cf, train, test)
cf_preds = cf_metrics.pop('predictions')
pd.Series({k: round(v, 4) for k, v in cf_metrics.items() if k != 'model'})


rmse            0.9616
mae             0.7513
precision@10    0.0616
recall@10       0.0143
dtype: float64

## 4. Model 3 — Matrix Factorization (SVD, k=40)

**Rationale:** Latent-factor model testing our core research hypothesis.


In [5]:
mf = MatrixFactorizationSVD()
mf.fit(train)
mf_metrics = evaluate_model(mf, train, test)
mf_preds = mf_metrics.pop('predictions')
pd.Series({k: round(v, 4) for k, v in mf_metrics.items() if k != 'model'})


rmse            0.9590
mae             0.7500
precision@10    0.0252
recall@10       0.0082
dtype: float64

## 5. Comparison table


In [6]:
results = pd.DataFrame([
    {'model': baseline.name, **{k: v for k, v in base_metrics.items()}},
    {'model': item_cf.name, **{k: v for k, v in cf_metrics.items()}},
    {'model': mf.name, **{k: v for k, v in mf_metrics.items()}},
])
results.round(4)


,model,rmse,mae,precision@10,recall@10
0,Bias Baseline (μ + user + item),0.9772,0.7662,0.0090,0.0031
1,Item-Based CF (k=30),0.9616,0.7513,0.0616,0.0143
2,"Matrix Factorization (SVD, k=40)",0.9590,0.7500,0.0252,0.0082


## 6. Stakeholder visualisations


In [7]:
plot_model_comparison(results, FIG_DIR)
plot_stakeholder_lift(results, FIG_DIR)
plot_prediction_quality(test, mf_preds, mf.name, FIG_DIR)
plot_recommendation_preview(mf, train, test, RAW_DIR, FIG_DIR)
print("Figures saved to", FIG_DIR)


Figures saved to C:\Users\admin\Desktop\数据科学项目\Work\M5\figures
